<a href="https://colab.research.google.com/github/JacquotQ/GDPR-compliance-with-Glass-Box/blob/main/Legalbert_final_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.28.0 datasets==2.16.1
!pip install scikit-learn==1.2.2 seaborn==0.12.2 accelerate==0.24.1 imbalanced-learn==0.10.1
!pip install numpy==1.25.2 pandas==2.0.3
!pip install evaluate



Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118


In [ ]:
from google.colab import drive

In [ ]:

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import time
import json
import warnings
import logging
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import torch.nn as nn

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('Violationresult')

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class WeightedLossTrainer(Trainer):
    """
    A custom Trainer that applies class weights when computing loss.
    """
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            logger.info(f"WeightedLossTrainer initialized with class weights: {class_weights}")
            self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(self.args.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Override compute_loss method.
        """
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

class TransformerClassifier:
    def __init__(self, model_name="JQ1984/legalbert_gdpr_pretrained", num_labels=None, output_dir="/content/drive/MyDrive/result", multi_label=False):
        self.model_name = model_name
        self.num_labels = num_labels
        self.output_dir = output_dir
        self.multi_label = multi_label
        self.tokenizer = None
        self.model = None
        self.trainer = None
        self.train_time = None

        # Training configuration
        self.training_config = {
            'model_name': model_name,
            'epochs': 30,
            'batch_size': 16,
            'learning_rate': 3e-5,
            'weight_decay': 0.01,
            'max_length': 256,
            'use_class_weights': True,
            'sampling_method': 'SMOTE',
            'sampling_rate': 'auto'
        }

        # Attributes for data handling
        self.label_encoder = None
        self.kf = None
        self.fold_datasets = []

        # Store all training data for final model
        self.all_train_texts = []
        self.all_train_labels = []

        # External test data attributes
        self.X_test_external_original = None
        self.y_test_external = None
        self.external_test_dataset = None

        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        # Baseline model output directory
        self.baseline_output_dir = os.path.join(self.output_dir, "baseline_model_results")
        if not os.path.exists(self.baseline_output_dir):
            os.makedirs(self.baseline_output_dir)
        self.baseline_model_last_fold = None

    def get_training_config(self):
        """Return current training configuration"""
        return self.training_config

    def _features_to_text(self, df):
        texts = []
        for _, row in df.iterrows():
            text = ""

            for col, val in row.items():
                if col == 'gdpr_clause' and isinstance(val, str):
                    clauses = [clause.strip() for clause in str(val).split(',')]
                    clause_text = " and ".join(clauses)
                    text += f"GDPR clauses are {clause_text}. "

                elif col == 'Date' and isinstance(val, str):
                    text += f"Date is {val}. "

                elif col in ['country', 'company_industry'] and isinstance(val, str):
                    text += f"{col} is {val}. "

                elif isinstance(val, (int, float)):
                    if val == 1:
                        feature_name = col.replace('_', ' ').lower()

                        if col in ['data_category_Children_data', 'data_category_Special_category_data']:
                            text += f"SENSITIVE DATA: {feature_name} is true. "

                        elif col in ['free_speech_exception', 'country_security_exception', 'Criminal_investigation_exception']:
                            text += f"EXCEPTION EXISTS: {feature_name} is true. "

                        else:
                            text += f"{feature_name} is true. "

                elif isinstance(val, str):
                    text += f"{col} is {val}. "

            texts.append(text)
        return texts

    def prepare_data(self, df, target_columns, n_splits=5, use_balance=True):
        """Prepare data for K-fold cross validation"""
        logger.info(f"Preparing dataset for {n_splits}-fold cross validation, target columns: {target_columns}")

        exclude_columns = ['fine_amount']

        if 'gdpr_clause' in df.columns:
            exclude_columns.append('gdpr_clause')

        df_copy = df.drop(columns=exclude_columns, errors='ignore')
        logger.info(f"Excluded columns: {exclude_columns}")

        if not target_columns:
            raise ValueError("Target columns not specified. Ensure the dataset contains a 'violation_result' column or provide a custom list of target columns.")

        missing_cols = [col for col in target_columns if col not in df_copy.columns]
        if missing_cols:
            raise ValueError(f"The following target columns are missing from the dataset after exclusions: {missing_cols}")

        for col in df_copy.select_dtypes(include=['object']).columns:
            df_copy[col] = df_copy[col].fillna('')
        for col in df_copy.select_dtypes(include=['number']).columns:
            if col not in target_columns:
                df_copy[col] = df_copy[col].fillna(df_copy[col].median())

        target_col = target_columns[0]

        if df_copy[target_col].dtype == 'object':
            logger.info(f"Target column '{target_col}' is categorical, converting to numeric categories")
            le = LabelEncoder()
            df_copy[target_col] = le.fit_transform(df_copy[target_col])
            self.label_encoder = le
            logger.info(f"Category mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

        X = df_copy.drop(columns=target_columns)
        Y = df_copy[target_col].astype('int64')

        # Encode categorical features for SMOTE
        label_encoders = {}
        X_encoded = X.copy()
        for col in X.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            X_encoded[col] = le.fit_transform(X[col])
            label_encoders[col] = le
            logger.info(f"Encoded categorical column '{col}' for SMOTE")

        if self.num_labels is None or self.num_labels != len(Y.unique()):
            if self.num_labels is not None and self.num_labels != len(Y.unique()):
                logger.warning(f"Initialized num_labels ({self.num_labels}) does not match unique values in target ({len(Y.unique())}). Updating.")
            self.num_labels = len(Y.unique())

        logger.info(f"Feature count (after exclusions for text generation): {X.shape[1]}, Sample count: {X.shape[0]}")
        logger.info(f"Category count: {self.num_labels}, Category distribution: {Y.value_counts().to_dict()}")

        X_text = self._features_to_text(X)

        # Store all texts and labels for final training
        self.all_train_texts = X_text
        self.all_train_labels = Y.values

        self.kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        self.fold_datasets = []

        for fold_idx, (train_idx, val_idx) in enumerate(self.kf.split(X_text)):
            if use_balance:
                X_train_fold_encoded = X_encoded.iloc[train_idx]
                y_train_fold = Y.iloc[train_idx]

                logger.info(f"Fold {fold_idx + 1} - Original training distribution: {Counter(y_train_fold)}")
                smote = SMOTE(random_state=42)
                X_train_fold_balanced, y_train_fold_balanced = smote.fit_resample(X_train_fold_encoded, y_train_fold)
                logger.info(f"Fold {fold_idx + 1} - Balanced training distribution: {Counter(y_train_fold_balanced)}")

                X_train_fold_balanced_df = pd.DataFrame(X_train_fold_balanced, columns=X_encoded.columns)

                X_train_fold_decoded = X_train_fold_balanced_df.copy()
                for col, le in label_encoders.items():
                    X_train_fold_decoded[col] = le.inverse_transform(X_train_fold_balanced_df[col].astype(int))

                X_train_fold_text = self._features_to_text(X_train_fold_decoded)
                y_train_fold = y_train_fold_balanced
            else:
                X_train_fold_text = [X_text[i] for i in train_idx]
                y_train_fold = Y.iloc[train_idx].values

            X_val_fold_text = [X_text[i] for i in val_idx]
            y_val_fold = Y.iloc[val_idx].values

            train_dataset = Dataset.from_dict({'text': X_train_fold_text, 'labels': y_train_fold})
            val_dataset = Dataset.from_dict({'text': X_val_fold_text, 'labels': y_val_fold})
            self.fold_datasets.append((train_dataset, val_dataset))

        logger.info(f"Created {n_splits} folds for cross-validation stored in self.fold_datasets")
        return True

    def load_external_test_data(self, file_path, target_columns):
        """Load external test dataset"""
        logger.info(f"Loading external test data from {file_path}")

        if file_path.endswith('.csv'):
            test_df = pd.read_csv(file_path, sep=';')
        else:
            raise ValueError("Unsupported file format")

        if 'Affected_data_volume' in test_df.columns:
            logger.info("Handling Affected_data_volume column")
            if test_df['Affected_data_volume'].dtype == 'object':
                test_df['Affected_data_volume'] = pd.to_numeric(
                    test_df['Affected_data_volume'].replace('unspecific', 0),
                    errors='coerce'
                ).fillna(0)

        for col in test_df.select_dtypes(include=['object']).columns:
            test_df[col] = test_df[col].fillna('')
        for col in test_df.select_dtypes(include=['number']).columns:
            if col not in target_columns:
                test_df[col] = test_df[col].fillna(test_df[col].median())

        target_col = target_columns[0]
        has_labels_in_file = False
        if target_col in test_df.columns:
            has_labels_in_file = True
            if hasattr(self, 'label_encoder') and self.label_encoder and test_df[target_col].dtype == 'object':
                test_df[target_col] = test_df[target_col].fillna(self.label_encoder.classes_[0])
                unknown_categories = set(test_df[target_col].unique()) - set(self.label_encoder.classes_)
                if unknown_categories:
                    logger.warning(f"Unknown categories found in the test set: {unknown_categories}")
                    mode_category = self.label_encoder.classes_[0]
                    for cat in unknown_categories:
                        test_df.loc[test_df[target_col] == cat, target_col] = mode_category
                test_df[target_col] = self.label_encoder.transform(test_df[target_col])

            self.y_test_external = test_df[target_col].astype('int64').values
        else:
            self.y_test_external = None

        X_test_df = test_df.drop(columns=[target_col] if target_col in test_df.columns else [], errors='ignore')
        self.X_test_external_original = self._features_to_text(X_test_df)

        if has_labels_in_file and self.y_test_external is not None:
            self.external_test_dataset = Dataset.from_dict(
                {'text': self.X_test_external_original, 'labels': self.y_test_external}
            )
        else:
            self.external_test_dataset = Dataset.from_dict({'text': self.X_test_external_original})

        logger.info(f"External test size: {len(self.external_test_dataset)}")
        return True

    def load_model(self):
        logger.info(f"Loading model: {self.model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.num_labels is None:
            raise ValueError("self.num_labels has not been set. Run prepare_data first.")
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="multi_label_classification" if self.multi_label else None
        )
        self.model.to(device)

    def tokenize_data(self, dataset, max_length=256):
        """Tokenize a single dataset"""
        if not self.tokenizer:
            logger.error("Tokenizer not available. Load model first.")
            return None
        def tokenize_function(examples):
            return self.tokenizer(
                examples['text'],
                padding="max_length",
                truncation=True,
                max_length=max_length
            )
        return dataset.map(tokenize_function, batched=True)

    def train_and_evaluate_kfold(self, epochs=30, batch_size=16, learning_rate=3e-5, weight_decay=0.01, class_weights=None, early_stopping_patience: int = 5):
        """Train and evaluate using K-fold cross validation"""
        logger.info("Starting K-fold cross validation training")

        # Update training config
        self.training_config['epochs'] = epochs
        self.training_config['batch_size'] = batch_size
        self.training_config['learning_rate'] = learning_rate
        self.training_config['weight_decay'] = weight_decay
        self.training_config['use_class_weights'] = class_weights is not None

        if class_weights is not None:
            logger.info(f"Using class weights for training: {class_weights}")

        self.load_model()

        fold_results = []
        fold_accuracies = []
        all_fold_train_times = []

        label_encoder_for_metrics = self.label_encoder

        for fold, (train_dataset, val_dataset) in enumerate(self.fold_datasets):
            logger.info(f"Training fold {fold+1}/{len(self.fold_datasets)}")

            fold_output_dir = os.path.join(self.output_dir, f"fold_{fold+1}")
            os.makedirs(fold_output_dir, exist_ok=True)

            tokenized_train = self.tokenize_data(train_dataset)
            tokenized_val = self.tokenize_data(val_dataset)

            if tokenized_train is None or tokenized_val is None:
                continue

            if fold > 0:
                self.model = AutoModelForSequenceClassification.from_pretrained(
                    self.model_name,
                    num_labels=self.num_labels,
                    problem_type="multi_label_classification" if self.multi_label else None,
                    from_tf=False,
                    local_files_only=True
                ).to(device)

            training_args = TrainingArguments(
                output_dir=fold_output_dir,
                num_train_epochs=epochs,
                per_device_train_batch_size=batch_size,
                per_device_eval_batch_size=batch_size,
                weight_decay=weight_decay,
                learning_rate=learning_rate,
                logging_dir=f"{fold_output_dir}/logs",
                evaluation_strategy="epoch",
                save_strategy="epoch",
                load_best_model_at_end=True,
                metric_for_best_model="f1",
                greater_is_better=True,
                push_to_hub=False,
                report_to="none",
                logging_steps=999999  # Suppress per-step logging
            )

            def compute_metrics(eval_pred):
                logits, labels = eval_pred
                preds = np.argmax(logits, axis=1)

                precision, recall, f1, _ = precision_recall_fscore_support(
                    labels, preds, average='macro', zero_division=0
                )
                acc = accuracy_score(labels, preds)

                return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

            current_trainer = WeightedLossTrainer(
                model=self.model,
                args=training_args,
                train_dataset=tokenized_train,
                eval_dataset=tokenized_val,
                compute_metrics=compute_metrics,
                tokenizer=self.tokenizer,
                class_weights=class_weights
            )

            start_time_fold = time.time()
            current_trainer.train()
            current_fold_train_time = time.time() - start_time_fold
            all_fold_train_times.append(current_fold_train_time)

            eval_results = current_trainer.evaluate()
            fold_accuracy = eval_results['eval_accuracy']
            fold_accuracies.append(fold_accuracy)

            # Generate confusion matrix for this fold
            val_predictions = current_trainer.predict(tokenized_val)
            val_preds = np.argmax(val_predictions.predictions, axis=1)
            val_labels = val_dataset['labels']
            cm_fold = confusion_matrix(val_labels, val_preds)

            logger.info(f"Fold {fold+1} training time: {current_fold_train_time:.2f}s, Accuracy: {fold_accuracy:.4f}, F1: {eval_results.get('eval_f1', 0.0):.4f}")
            logger.info(f"Fold {fold+1} Confusion Matrix:\n{cm_fold}")

            # Save fold results including confusion matrix
            fold_results_with_cm = {
                **eval_results,
                'confusion_matrix': cm_fold.tolist()
            }
            with open(os.path.join(fold_output_dir, 'eval_results.json'), 'w') as f:
                json.dump(fold_results_with_cm, f)

            fold_results.append({
                'fold': fold+1,
                'eval_results': eval_results,
                'confusion_matrix': cm_fold,
                'training_time': current_fold_train_time
            })
            current_trainer.save_model(os.path.join(fold_output_dir, "best_model"))
            if fold == len(self.fold_datasets) - 1:
                self.trainer = current_trainer

        print("\n===== K-fold Cross Validation Results =====")
        for i, acc in enumerate(fold_accuracies):
            print(f"Fold {i+1} Accuracy: {acc:.4f}")
        print("==========================================")

        avg_results = {
            'avg_accuracy': np.mean(fold_accuracies) if fold_accuracies else 0.0,
            'avg_f1': np.mean([res['eval_results'].get('eval_f1', 0.0) for res in fold_results if 'eval_results' in res]) if fold_results else 0.0,
            'avg_precision': np.mean([res['eval_results'].get('eval_precision', 0.0) for res in fold_results if 'eval_results' in res]) if fold_results else 0.0,
            'avg_recall': np.mean([res['eval_results'].get('eval_recall', 0.0) for res in fold_results if 'eval_results' in res]) if fold_results else 0.0,
            'avg_training_time': np.mean(all_fold_train_times) if all_fold_train_times else 0.0
        }
        with open(os.path.join(self.output_dir, 'avg_kfold_results.json'), 'w') as f:
            json.dump(avg_results, f)
        logger.info(f"K-fold cross validation complete. Average Accuracy: {avg_results['avg_accuracy']:.4f}, Average F1: {avg_results['avg_f1']:.4f}")

        # Save training configuration
        with open(os.path.join(self.output_dir, 'training_config.json'), 'w') as f:
            json.dump(self.training_config, f)

        return fold_results, avg_results, fold_accuracies

    def train_final_model_on_all_data(self, epochs=30, batch_size=16, learning_rate=3e-5, weight_decay=0.01, class_weights=None):
        """Train final model on all training data"""
        logger.info("Training final model on all training data")

        if not self.all_train_texts or len(self.all_train_texts) == 0:
            logger.error("No training data available. Run prepare_data first.")
            return None

        # Create dataset from all training data
        full_dataset = Dataset.from_dict({
            'text': self.all_train_texts,
            'labels': self.all_train_labels
        })

        logger.info(f"Full dataset size: {len(full_dataset)}")
        logger.info(f"Label distribution in full dataset: {Counter(self.all_train_labels)}")

        # Load fresh model
        self.load_model()

        # Tokenize full dataset
        tokenized_full = self.tokenize_data(full_dataset)
        if tokenized_full is None:
            logger.error("Failed to tokenize full dataset")
            return None

        final_output_dir = os.path.join(self.output_dir, "final_model")
        os.makedirs(final_output_dir, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=final_output_dir,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=weight_decay,
            learning_rate=learning_rate,
            logging_dir=f"{final_output_dir}/logs",
            save_strategy="epoch",
            evaluation_strategy="no",  # No evaluation during training
            load_best_model_at_end=False,
            push_to_hub=False,
            report_to="none",
            logging_steps=999999
        )

        # Create trainer for final model
        final_trainer = WeightedLossTrainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_full,
            tokenizer=self.tokenizer,
            class_weights=class_weights
        )

        # Train final model
        start_time = time.time()
        final_trainer.train()
        final_train_time = time.time() - start_time

        logger.info(f"Final model training completed in {final_train_time:.2f}s")

        # Evaluate on full training set
        predictions = final_trainer.predict(tokenized_full)
        preds = np.argmax(predictions.predictions, axis=1)

        # Calculate metrics
        accuracy = accuracy_score(self.all_train_labels, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            self.all_train_labels, preds, average='macro', zero_division=0
        )

        # Generate confusion matrix
        cm_final = confusion_matrix(self.all_train_labels, preds)

        # Generate classification report
        target_names = self.label_encoder.classes_ if self.label_encoder is not None else None
        report_dict = classification_report(
            self.all_train_labels, preds,
            output_dict=True,
            zero_division=0,
            target_names=target_names
        )
        report_df = pd.DataFrame(report_dict).transpose()
        report_df.to_csv(os.path.join(final_output_dir, 'final_model_classification_report.csv'))

        print("\n===== Final Model Performance on Full Training Data =====")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Macro F1: {f1:.4f}")
        print(f"Macro Precision: {precision:.4f}")
        print(f"Macro Recall: {recall:.4f}")
        print(f"\nConfusion Matrix:")
        print(cm_final)
        print("\nClassification Report:")
        print(report_df)
        print("========================================================")

        # Save results
        final_results = {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'confusion_matrix': cm_final.tolist(),
            'training_time': final_train_time,
            'dataset_size': len(full_dataset)
        }

        with open(os.path.join(final_output_dir, 'final_model_results.json'), 'w') as f:
            json.dump(final_results, f)

        # Save the final model
        final_trainer.save_model(os.path.join(final_output_dir, "best_model"))

        # Update the trainer to use final model for external evaluation
        self.trainer = final_trainer

        return final_results

    def evaluate_external_test(self):
        """Evaluate on the external test set using the last fold's model"""
        logger.info("Evaluating model on external test set")

        if not hasattr(self, 'external_test_dataset') or self.external_test_dataset is None:
            logger.error("No external test dataset loaded")
            return None

        if not self.trainer:
            logger.error("No trainer available. Run train_and_evaluate_kfold first.")
            return None

        tokenized_test = self.tokenize_data(self.external_test_dataset)
        if tokenized_test is None:
            logger.error("Tokenization of external test set failed.")
            return None

        has_labels_in_dataset = 'labels' in self.external_test_dataset.features

        if has_labels_in_dataset and self.trainer:
            test_predictions_output = self.trainer.predict(tokenized_test)
            preds = np.argmax(test_predictions_output.predictions, axis=1)

            if self.y_test_external is None:
                logger.error("self.y_test_external is None, cannot calculate external test metrics.")
                return {'predictions': preds}

            cm = confusion_matrix(self.y_test_external, preds)
            report_dict = classification_report(
                self.y_test_external,
                preds,
                output_dict=True,
                zero_division=0,
                target_names=self.label_encoder.classes_ if hasattr(self, 'label_encoder') and self.label_encoder else None
            )
            report_df = pd.DataFrame(report_dict).transpose()
            report_df.to_csv(os.path.join(self.output_dir, 'external_test_classification_report.csv'))

            print("\nExternal Test Set Classification Report:")
            print(report_df)

            return {
                'test_results': test_predictions_output.metrics,
                'confusion_matrix': cm,
                'classification_report': report_df
            }
        elif self.trainer:
            logger.info("External test data has no labels in dataset features or self.y_test_external is None. Generating predictions only.")
            test_predictions_output = self.trainer.predict(tokenized_test)
            preds = np.argmax(test_predictions_output.predictions, axis=1)
            pred_df = pd.DataFrame({'prediction': preds})
            pred_df.to_csv(os.path.join(self.output_dir, 'external_test_predictions.csv'), index=False)
            print("\nPredictions for external test set saved to 'external_test_predictions.csv'")
            return {'predictions': preds}
        else:
            logger.error("No trainer available for prediction.")
            return None

    # --- BASELINE MODEL METHODS ---
    def train_and_evaluate_baseline_kfold(self):
        """Train and evaluate baseline model (TF-IDF + Logistic Regression) using K-fold cross validation."""
        logger.info("Starting K-fold cross validation for Baseline model")

        if not self.fold_datasets:
            logger.error("Fold datasets not prepared for baseline. Run prepare_data first.")
            return [], {}, []

        fold_results_baseline = []
        fold_accuracies_baseline = []
        all_train_times_baseline = []

        actual_n_splits = len(self.fold_datasets)
        if actual_n_splits == 0:
            logger.error("No fold datasets available for baseline K-fold.")
            return [], {}, []

        logger.info(f"Baseline K-fold will run for {actual_n_splits} folds.")

        for fold, (train_hf_dataset, val_hf_dataset) in enumerate(self.fold_datasets):
            logger.info(f"Training baseline fold {fold+1}/{actual_n_splits}")

            X_train_text = train_hf_dataset['text']
            y_train = np.array(train_hf_dataset['labels'])
            X_val_text = val_hf_dataset['text']
            y_val = np.array(val_hf_dataset['labels'])

            pipeline = make_pipeline(
                TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95),
                LogisticRegression(solver='liblinear', random_state=(42 + fold), C=1.0, class_weight='balanced')
            )

            start_time = time.time()
            try:
                pipeline.fit(X_train_text, y_train)
            except Exception as e:
                logger.error(f"Error fitting baseline pipeline for fold {fold+1}: {e}")
                fold_accuracies_baseline.append(0.0)
                fold_results_baseline.append({
                    'fold': fold+1,
                    'eval_results': {'eval_accuracy': 0.0, 'eval_f1': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0},
                    'training_time': 0, 'status': 'training_error'
                })
                continue

            train_time_fold = time.time() - start_time
            all_train_times_baseline.append(train_time_fold)

            preds_val = pipeline.predict(X_val_text)

            acc_val = accuracy_score(y_val, preds_val)
            precision_val, recall_val, f1_val, _ = precision_recall_fscore_support(y_val, preds_val, average='macro', zero_division=0)

            fold_accuracies_baseline.append(acc_val)
            eval_results_fold_baseline = {'eval_accuracy': acc_val, 'eval_f1': f1_val, 'eval_precision': precision_val, 'eval_recall': recall_val}

            logger.info(f"Baseline Fold {fold+1} training time: {train_time_fold:.2f}s, Accuracy: {acc_val:.4f}")

            fold_output_dir_bl = os.path.join(self.baseline_output_dir, f"fold_{fold+1}")
            os.makedirs(fold_output_dir_bl, exist_ok=True)
            with open(os.path.join(fold_output_dir_bl, 'eval_results_baseline.json'), 'w') as f:
                json.dump(eval_results_fold_baseline, f)

            fold_results_baseline.append({
                'fold': fold+1,
                'eval_results': eval_results_fold_baseline,
                'training_time': train_time_fold,
                'status': 'success'
            })

            if fold == actual_n_splits - 1:
                self.baseline_model_last_fold = pipeline
                logger.info(f"Saved baseline pipeline from fold {fold+1} to self.baseline_model_last_fold.")

        avg_accuracy_baseline = np.mean([acc for acc in fold_accuracies_baseline if isinstance(acc, float)]) if fold_accuracies_baseline else 0.0
        logger.info(f"Baseline K-fold cross validation complete. Average Accuracy: {avg_accuracy_baseline:.4f}")

        avg_results_baseline_dict = {
            'avg_accuracy': avg_accuracy_baseline,
            'avg_f1': np.mean([res['eval_results'].get('eval_f1', 0.0) for res in fold_results_baseline if res.get('status') == 'success' and 'eval_results' in res]) if fold_results_baseline else 0.0,
            'avg_precision': np.mean([res['eval_results'].get('eval_precision', 0.0) for res in fold_results_baseline if res.get('status') == 'success' and 'eval_results' in res]) if fold_results_baseline else 0.0,
            'avg_recall': np.mean([res['eval_results'].get('eval_recall', 0.0) for res in fold_results_baseline if res.get('status') == 'success' and 'eval_results' in res]) if fold_results_baseline else 0.0,
            'avg_training_time': np.mean(all_train_times_baseline) if all_train_times_baseline else 0.0
        }
        with open(os.path.join(self.baseline_output_dir, 'avg_kfold_results_baseline.json'), 'w') as f:
            json.dump(avg_results_baseline_dict, f)

        return fold_results_baseline, avg_results_baseline_dict, fold_accuracies_baseline

    def evaluate_baseline_external_test(self):
        """Evaluate the baseline model on the external test set."""
        logger.info("Evaluating Baseline model on external test set")
        if self.X_test_external_original is None:
            logger.error("No external test data text (X_test_external_original) available for Baseline.")
            return None
        if self.baseline_model_last_fold is None:
            logger.error("No Baseline model (pipeline) available from K-fold training (self.baseline_model_last_fold is None).")
            return None

        X_test_text = self.X_test_external_original

        if self.y_test_external is not None:
            logger.info("External test set has labels. Evaluating Baseline model with metrics.")
            try:
                preds_test_baseline = self.baseline_model_last_fold.predict(X_test_text)
            except Exception as e:
                logger.error(f"Error predicting with baseline model on external test data: {e}")
                return None

            acc_test = accuracy_score(self.y_test_external, preds_test_baseline)
            precision_test, recall_test, f1_test, _ = precision_recall_fscore_support(self.y_test_external, preds_test_baseline, average='macro', zero_division=0)

            test_metrics_baseline = {
                'test_accuracy': acc_test,
                'test_f1': f1_test,
                'test_precision': precision_test,
                'test_recall': recall_test
            }
            cm_baseline = confusion_matrix(self.y_test_external, preds_test_baseline)
            target_names_for_report = None
            if hasattr(self, 'label_encoder') and self.label_encoder is not None:
                try:
                    max_label_idx = max(np.max(self.y_test_external), np.max(preds_test_baseline))
                    if max_label_idx < len(self.label_encoder.classes_):
                        target_names_for_report = self.label_encoder.classes_
                    else:
                        logger.warning(f"Max label index ({max_label_idx}) for baseline external test is out of bounds for label encoder classes (size {len(self.label_encoder.classes_)}). Reporting without target names.")
                except Exception as e_classes:
                    logger.warning(f"Could not determine target names for baseline classification report: {e_classes}")

            report_dict_baseline = classification_report(
                self.y_test_external, preds_test_baseline, output_dict=True, zero_division=0,
                target_names=target_names_for_report
            )
            report_df_baseline = pd.DataFrame(report_dict_baseline).transpose()
            report_df_baseline.to_csv(os.path.join(self.baseline_output_dir, 'baseline_external_test_classification_report.csv'))
            logger.info(f"\nBaseline External Test Set Classification Report:\n{report_df_baseline}")
            logger.info(f"Baseline External Test Metrics: {test_metrics_baseline}")
            return {'test_metrics': test_metrics_baseline, 'confusion_matrix': cm_baseline.tolist(), 'classification_report_df': report_df_baseline.to_dict()}
        else:
            logger.info("External test set has no labels. Generating predictions with Baseline model.")
            try:
                preds_test_baseline = self.baseline_model_last_fold.predict(X_test_text)
                pd.DataFrame({'prediction': preds_test_baseline}).to_csv(os.path.join(self.baseline_output_dir, 'baseline_external_test_predictions.csv'), index=False)
                logger.info("Baseline predictions for external test set saved.")
                return {'predictions': preds_test_baseline.tolist()}
            except Exception as e:
                logger.error(f"Error predicting with baseline model on external test data (no labels): {e}")
                return None

In [ ]:
from collections import Counter

if __name__ == '__main__':
    # Define file paths
    train_file_path = '/content/drive/MyDrive/Thesis/FINALFI.csv'
    test_file_path = '/content/drive/MyDrive/Thesis/Testdataset.csv'

    # Load training data
    try:
        df = pd.read_csv(train_file_path, sep=';')
    except FileNotFoundError:
        logger.error(f"Training file not found at {train_file_path}. Please check the path.")
        exit()
    except Exception as e:
        logger.error(f"Error reading training file: {e}")
        exit()

    target_columns = ['violation_result']

    # ==============================================================================
    # Calculate class weights and pass to training function
    # ==============================================================================

    # 1. Calculate class weights
    # We use the original labels (before SMOTE) to calculate weights
    logger.info("Calculating class weights for weighted loss...")
    le_for_weights = LabelEncoder()
    y_labels = le_for_weights.fit_transform(df[target_columns[0]])
    class_names = le_for_weights.classes_

    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_labels),
        y=y_labels
    )
    logger.info(f"Detected Classes: {class_names}")
    logger.info(f"Calculated Class Weights (for [0, 1]): {weights}")

    # 2. Create classifier instance
    # Modified output directory to avoid overwriting previous results
    classifier = TransformerClassifier(
        model_name="JQ1984/legalbert_gdpr_pretrained",
        output_dir="/content/drive/MyDrive/Thesis/violation_result_model_weighted",
        multi_label=False
    )

    # 3. Prepare data (SMOTE can still be used as auxiliary)
    if not classifier.prepare_data(df, target_columns, n_splits=5, use_balance=True):
        logger.error("Data preparation failed. Exiting.")
        exit()

    # 4. Load external test set
    try:
        if not classifier.load_external_test_data(test_file_path, target_columns):
            logger.warning("Loading external test data failed or file not found.")
    except FileNotFoundError:
        logger.warning(f"Test file not found at {test_file_path}. External evaluation will be skipped.")
    except Exception as e:
        logger.warning(f"Error loading external test data: {e}.")

    # 5. Train with K-fold cross validation
    logger.info("\n--- Starting Transformer Model K-Fold Cross Validation (with Weighted Loss) ---")
    transformer_fold_results, transformer_avg_results, transformer_fold_accuracies = classifier.train_and_evaluate_kfold(
        epochs=10,
        batch_size=16,
        learning_rate=3e-5,
        weight_decay=0.01,
        class_weights=weights  # Key: pass calculated weights
    )

    # Print Transformer K-fold accuracies
    print("\n===== Transformer K-fold Cross Validation Summary =====")
    print("\nPer-fold Accuracies:")
    for i, acc in enumerate(transformer_fold_accuracies):
        print(f"  Fold {i+1}: {acc:.4f}")

    # Print Transformer average results
    print("\nAverage Cross Validation Metrics:")
    for metric, value in transformer_avg_results.items():
        if isinstance(value, float):
            print(f"  {metric}: {value:.4f}")

    # ==============================================================================
    # NEW: Train final model on all data
    # ==============================================================================
    logger.info("\n--- Training Final Model on Complete Dataset ---")
    final_results = classifier.train_final_model_on_all_data(
        epochs=10,
        batch_size=16,
        learning_rate=3e-5,
        weight_decay=0.01,
        class_weights=weights
    )

    if final_results:
        print("\n===== Final Model Training Complete =====")
        print(f"Final model saved in: {classifier.output_dir}/final_model/")

    # ==============================================================================
    # Evaluate on external test set
    # ==============================================================================
    logger.info("\n--- Evaluating Transformer Model on External Test Set ---")
    transformer_test_results = classifier.evaluate_external_test()

    if transformer_test_results and 'confusion_matrix' in transformer_test_results:
        print("\n===== External Test Set Evaluation Complete =====")
        print("Confusion Matrix for External Test Set:")
        print(transformer_test_results['confusion_matrix'])

    # ==============================================================================
    # (Optional) Train and evaluate baseline model
    # ==============================================================================
    logger.info("\n--- Starting Baseline Model K-Fold Cross Validation ---")
    baseline_fold_results, baseline_avg_results, baseline_fold_accuracies = classifier.train_and_evaluate_baseline_kfold()

    print("\n===== Baseline K-fold Cross Validation Summary =====")
    print(f"Average Accuracy: {baseline_avg_results['avg_accuracy']:.4f}")

    # Evaluate baseline on external test set
    logger.info("\n--- Evaluating Baseline Model on External Test Set ---")
    baseline_test_results = classifier.evaluate_baseline_external_test()
    if baseline_test_results and 'test_metrics' in baseline_test_results:
        print("\nBaseline External Test Metrics:")
        for metric, value in baseline_test_results['test_metrics'].items():
            print(f"  {metric}: {value:.4f}")

    # ==============================================================================
    # Display training configuration
    # ==============================================================================
    print("\n===== Training Configuration =====")
    config = classifier.get_training_config()
    for param, value in config.items():
        print(f"  {param}: {value}")

    print(f"\n===== All Results Saved =====")
    print(f"Output directory: {classifier.output_dir}")
    print(f"- K-fold results: {classifier.output_dir}/fold_*/")
    print(f"- Final model: {classifier.output_dir}/final_model/")
    print(f"- Configuration: {classifier.output_dir}/training_config.json")
    print(f"- Average results: {classifier.output_dir}/avg_kfold_results.json")
    if baseline_fold_results:
        print(f"- Baseline results: {classifier.baseline_output_dir}/")

    print("\nTraining and evaluation complete!")

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/702k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss



===== Final Model Performance on Full Training Data =====
Accuracy: 0.9378
Macro F1: 0.8741
Macro Precision: 0.8796
Macro Recall: 0.8688

Confusion Matrix:
[[ 273   81]
 [  69 1989]]

Classification Report:
              precision    recall  f1-score      support
0              0.798246  0.771186  0.784483   354.000000
1              0.960870  0.966472  0.963663  2058.000000
accuracy       0.937811  0.937811  0.937811     0.937811
macro avg      0.879558  0.868829  0.874073  2412.000000
weighted avg   0.937002  0.937811  0.937365  2412.000000

===== Final Model Training Complete =====
Final model saved in: /content/drive/MyDrive/Thesis/violation_result_model_weighted/final_model/


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


External Test Set Classification Report:
              precision    recall  f1-score  support
0              0.428571  0.250000  0.315789    12.00
1              0.903226  0.954545  0.928177    88.00
accuracy       0.870000  0.870000  0.870000     0.87
macro avg      0.665899  0.602273  0.621983   100.00
weighted avg   0.846267  0.870000  0.854690   100.00

===== External Test Set Evaluation Complete =====
Confusion Matrix for External Test Set:
[[ 3  9]
 [ 4 84]]

===== Baseline K-fold Cross Validation Summary =====
Average Accuracy: 0.9154

Baseline External Test Metrics:
  test_accuracy: 0.8800
  test_f1: 0.6337
  test_precision: 0.7021
  test_recall: 0.6080

===== Training Configuration =====
  model_name: JQ1984/legalbert_gdpr_pretrained
  epochs: 10
  batch_size: 16
  learning_rate: 3e-05
  weight_decay: 0.01
  max_length: 256
  use_class_weights: True
  sampling_method: SMOTE
  sampling_rate: auto

===== All Results Saved =====
Output directory: /content/drive/MyDrive/Thesis/vi

In [ ]:
import torch
import pandas as pd
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader

class SimpleGDPRDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

def features_to_text(df):
    """转换特征为文本 - 与训练代码一致"""
    texts = []
    for _, row in df.iterrows():
        text = ""
        for col, val in row.items():
            if col == 'gdpr_clause' and isinstance(val, str):
                clauses = [clause.strip() for clause in str(val).split(',')]
                clause_text = " and ".join(clauses)
                text += f"GDPR clauses are {clause_text}. "
            elif col == 'Date' and isinstance(val, str):
                text += f"Date is {val}. "
            elif col in ['country', 'company_industry'] and isinstance(val, str):
                text += f"{col} is {val}. "
            elif isinstance(val, (int, float)):
                if val == 1:
                    feature_name = col.replace('_', ' ').lower()
                    if col in ['data_category_Children_data', 'data_category_Special_category_data']:
                        text += f"SENSITIVE DATA: {feature_name} is true. "
                    elif col in ['free_speech_exception', 'country_security_exception', 'Criminal_investigation_exception']:
                        text += f"EXCEPTION EXISTS: {feature_name} is true. "
                    else:
                        text += f"{feature_name} is true. "
            elif isinstance(val, str):
                text += f"{col} is {val}. "
        texts.append(text)
    return texts

def get_predictions():
    train_file_path = '/content/drive/MyDrive/Thesis/FINALFI.csv'
    test_file_path = '/content/drive/MyDrive/Thesis/Testdataset.csv'
    model_dir = "/content/drive/MyDrive/Thesis/violation_result_model_weighted/final_model/best_model"
    target_columns = ['violation_result']


    try:
        df = pd.read_csv(test_file_path, sep=';')
    except:
        df = pd.read_csv(test_file_path, sep=',')


    exclude_columns = ['fine_amount']
    exclude_columns.extend(target_columns)  # 排除目标列

    if 'gdpr_clause' in df.columns:
        exclude_columns.append('gdpr_clause')


    if 'Affected_data_volume' in df.columns:
        if df['Affected_data_volume'].dtype == 'object':
            df['Affected_data_volume'] = pd.to_numeric(
                df['Affected_data_volume'].replace('unspecific', 0),
                errors='coerce'
            ).fillna(0)

    df_processed = df.drop(columns=exclude_columns, errors='ignore')


    for col in df_processed.select_dtypes(include=['object']).columns:
        df_processed[col] = df_processed[col].fillna('')
    for col in df_processed.select_dtypes(include=['number']).columns:
        df_processed[col] = df_processed[col].fillna(df_processed[col].median())


    texts = features_to_text(df_processed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    dataset = SimpleGDPRDataset(texts, tokenizer)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

    all_predictions = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1)
            all_predictions.extend(predictions.cpu().numpy().tolist())

    result = {"predictions": all_predictions}
    print(json.dumps(result))

if __name__ == "__main__":
    get_predictions()

{"predictions": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]}
